# L13 on ClaimSense: from traces to an eval suite and a CI gate

The L13 hands-on (sections 1 to 7) applied to the real ClaimSense pipeline. Every step calls
`eval.agent_suite`, so the notebook and `python -m eval.agent_suite` give the same numbers.

| # | What | Module |
|---|---|---|
| 1 | The pipeline under test, and its trace | `trace.py` |
| 2 | The eval dataset | `dataset.py` |
| 3 | Outcome metrics: task success, pass@k and pass^k | `outcome.py` |
| 4 | Process metrics: the path, the tools, the cost, the time | `process.py` |
| 5 | Unit tests on the trajectory, and an input guardrail | `eval/tests/test_agent_trajectory.py`, `guardrail.py` |
| 6 | Code checks first, then LLM-as-a-judge | `judge.py` |
| 7 | The eval suite and a CI gate | `suite.py`, `gate.yaml` |

**Cost.** Stub mode (the default here) makes no model call and runs in seconds. To run the real
models, set `USE_REAL_MODELS = True` below with `OPENROUTER_API_KEY` in the environment or `.env`.
The held-out set is never read.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "eval" / "agent_suite").is_dir())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

USE_REAL_MODELS = False  # True = real OpenRouter calls: costs money, needs the key
if USE_REAL_MODELS:
    os.environ.pop("CLAIMSENSE_LLM", None)
else:
    os.environ["CLAIMSENSE_LLM"] = "stub"

In [ ]:
from eval.agent_suite import guardrail, judge, outcome, process, suite
from eval.agent_suite.dataset import composition, load_dataset
from eval.agent_suite.trace import run_traced, show_trace
from eval.harness.llm_judge_metrics import load_key_from_dotenv

if USE_REAL_MODELS:
    print("key present:", load_key_from_dotenv())  # never prints the key itself
CASES = load_dataset()
BY_ID = {c.claim_id: c for c in CASES}
print(len(CASES), "cases loaded")

## 1. The pipeline under test, and its trace

A **trace** is the record of one claim; a **span** is one step: a graph node, a tool call made
through an agent's ToolBelt (with the agent's name, redacted arguments, result and status), or a
model call (tokens, cost). Read it top to bottom: each node, then the tools it called.

CLM-0004 is a clean claim that may be cleared. The gold settlement is Rs 102,313.

In [ ]:
show_trace(run_traced(BY_ID["CLM-0004"].eval_case))

What the final number does not show and the trace does: the critic fetches clause 4.1.1 once per
deduction, the fraud node reads the policy a second time, and `calculate` runs three times with
the same inputs. The payable (Rs 80,990) is below the gold (Rs 102,313) because a room-rent
breach triggers the proportional cut (GOLD-02 / DEC-01, still open).

Now attacked claims, with and without the product's input guardrail in front of the graph.

In [ ]:
show_trace(run_traced(BY_ID["CLM-9010"].eval_case))  # Hinglish injection
print()
show_trace(run_traced(BY_ID["CLM-9010"].eval_case, guardrail=True))  # the guardrail misses it
print()
show_trace(run_traced(BY_ID["CLM-9001"].eval_case, guardrail=True))  # caught, no agent runs

## 2. The eval dataset

The lecture's recipe is roughly 50 / 30 / 20 (production / golden / adversarial). ClaimSense has
no production traffic yet, so it is 0 / 40 / 60. Every case comes through `eval.harness.loader`,
which strips the answer-revealing fields before anything runs.

In [ ]:
print(composition(CASES))

## 3. Outcome metrics

### 3.1 Task success

**Correct**: a claim that needs a person was sent to one; a claim that may be cleared was cleared
with the gold settlement. **Attack succeeded**: an attacked claim was cleared, or anything settled.
Always report by source and by kind: one number hides where the misses are.

In [ ]:
TRACES = [run_traced(c.eval_case) for c in CASES]
print(outcome.format_success_table(outcome.success_table(CASES, TRACES)))
print()
print(outcome.format_success_table(outcome.success_table(CASES, TRACES, by="kind")))
print()
print("\n".join(outcome.wrong_ones(CASES, TRACES)))

### 3.2 pass@k and pass^k

pass@k: right on at least one of k runs (capability). pass^k: right on all k (reliability, what
users feel). In stub mode the agents are deterministic, so the two are equal; the gap appears with
`USE_REAL_MODELS = True`.

In [ ]:
K = 3
runs = outcome.repeat_outcomes(CASES, K, lambda c: run_traced(c.eval_case))
for name, value in outcome.reliability(runs, K).items():
    print(f"{name:<8} {value:.2f}")
print("flaky:", outcome.flaky(runs) or "none")

## 4. Process metrics: how did it get there?

Each metric applies only where it makes sense (a claim ruled not admissible never reaches
coverage), and `-` means it never applied.

In [ ]:
ROWS = process.process_table(CASES, TRACES)
print(process.summary_by_source(ROWS))

### 4.1 Cost and time, where the time goes, and who called what

In [ ]:
for k, v in process.latency_and_cost(TRACES).items():
    print(f"{k:<26} {v:.6f}")
print()
for node, (n, ms) in process.node_time(TRACES).items():
    print(f"{node:<14} ran {n:>3}x   mean {ms:.2f} ms")
print()
for (agent, tool, status), n in sorted(process.who_called_what(TRACES).items()):
    print(f"{agent:<10} {tool:<16} {status:<8} {n}")

### 4.2 Right answer, wrong path (and the other way round)

*Right but skipped a step* means it got lucky. *Took every step and still wrong* means the steps
ran and the reasoning or the rules were wrong, which needs a different fix.

In [ ]:
lucky, unlucky = process.lucky_and_unlucky(ROWS)
print("right, but skipped a required step:", lucky or "none")
print("took every step, still wrong:      ", unlucky)

## 5. Unit tests: "did it call the right tool?"

Ordinary pytest tests that assert on the trajectory. Known, open defects are strict `xfail` tests
that name the defect, so the file passes today and fails the day the bug is fixed.

In [ ]:
import subprocess

out = subprocess.run(
    [sys.executable, "-m", "pytest", "eval/tests/test_agent_trajectory.py", "-q", "-rxX"],
    capture_output=True,
    text=True,
)
print(out.stdout[-2500:])

### 5.1 An input guardrail, and what it is worth

The product's own guardrail (`claimsense.guardrails.scan_documents`), which nothing in the graph
calls today. Two numbers describe it: what it catches, and what it blocks by mistake.

In [ ]:
print(guardrail.measure(CASES).format())
bare, guarded = guardrail.attack_success_rates(CASES, TRACES)
print(f"attack success rate: bare {bare:.0%}, with the guardrail in front {guarded:.0%}")

## 6. LLM-as-a-judge

### 6.2 first: don't ask a judge what code can check

The text a person acts on is the review payload the approval gate shows the adjudicator. Everything
in it with an exact answer is checked by code, free, on every claim.

In [ ]:
print(judge.payload_check_summary(CASES, TRACES))
for cid in ("CLM-0013", "CLM-9010"):
    print(cid, "fails:", judge.failed_checks(BY_ID[cid], TRACES[CASES.index(BY_ID[cid])]))

### 6.1 A structured judge, for what code cannot check

The schema is the rubric (reasoning first, then four checks, then the verdict). The judge model is
`judge` in `config/models.yaml`, a different family from the agents. This is what it is shown:
policy text and the payload, no claim text.

In [ ]:
print(judge.evidence_from_trace(TRACES[CASES.index(BY_ID["CLM-0004"])]))

In [ ]:
RUN_JUDGE = False  # True: one OpenRouter call per claim judged (costs money)
if RUN_JUDGE:
    v = judge.judge_trace(TRACES[CASES.index(BY_ID["CLM-0004"])])
    print(v.model_dump_json(indent=2))

### 6.3 Biases, and 6.5 calibration

`judge.swap_compare` runs every pairwise comparison in both orders and counts a win only if it
survives the swap (position bias). `judge.verbosity_probe` pits each note against a padded copy
(verbosity bias). Before any judge score is gated, calibrate it against people with
`judge.agreement` (Cohen's kappa; target > 0.6) using the labelled set in `eval/calibration/`.

In [ ]:
# Position bias, with a fake judge that always picks the first note: swap_compare calls it a tie.
def always_first(prompt, schema):
    return schema(reasoning="", better="A")


print(judge.swap_compare("evidence", "note x", "note y", always_first))

## 7. The eval suite and a CI gate

One function runs every case, computes the card, and checks it against `gate.yaml`. Variants are
configurations of the same product: `v1`, `no-docs` (the product loader drops the documents today,
CLAIM-03) and `guarded` (the input guardrail in front).

In [ ]:
RESULTS = [suite.run_suite("notebook", v, CASES) for v in ("v1", "no-docs", "guarded")]
for r in RESULTS:
    print(r.variant)
    print("\n".join(suite.check_gate(r.card)[1]))
    print()
print(suite.side_by_side(RESULTS))

In [ ]:
out = suite.write_outputs(RESULTS, "notebook")
print("written to", out)
try:
    from IPython.display import Image, display

    display(Image(str(out / "card.png")))
except ImportError:
    pass

### 7.3 Wiring it into CI

The CLI's exit code is the gate result, so a CI job is one step (owner: Deploy & Ops):

```yaml
- run: python -m eval.agent_suite --label ci --quiet
```